In [ ]:
import os
import glob
import csv
import math
import json
import sys
import pandas as pd
import numpy as np

sys.path.append("/Proyecto/Value-disagreement/Python/Utilities")
import Dict_Object

In [ ]:
# CONFIG
INPUT_DIR = "/Proyecto/Value-disagreement/Python/Models/Inference/deba_usr_comments"     # folder containing the 10 per-value CSVs
OUTPUT_DIR = "/Proyecto/Value-disagreement/Python/Models/Inference/agg_by_valuefile"     # to write per-value author summaries
CHUNK_SIZE = 500_000

FILE_GLOB = "*.csv"

In [ ]:
# HELPERS
def safe_upper_strip(x):
    try:
        return str(x).upper().strip()
    except Exception:
        return ""

def finalize_stats(agg_dict):
    """
    Convert running aggregates into final metrics:
      - n_comments_value
      - pos_count_value
      - prev_value
      - mean_prob_value
      - std_prob_value
    """
    rows = []
    for author, stats in agg_dict.items():
        n = stats["n"]
        pos = stats["pos"]
        sum_prob = stats["sum_prob"]
        sum_prob_sq = stats["sum_prob_sq"]

        prev = pos / n if n > 0 else 0.0
        mean = sum_prob / n if n > 0 else 0.0
        # population std (set ddof=0); use sample std if you prefer (ddof=1 when n>1)
        var = (sum_prob_sq / n - mean * mean) if n > 0 else 0.0
        var = max(var, 0.0)  # numerical safety
        std = math.sqrt(var)

        rows.append((author, n, pos, prev, mean, std))
    return rows

def write_summary(value_name, rows, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    # Generate compact, value-specific column names
    v = value_name
    out_path = os.path.join(out_dir, f"author_summary_{v}.csv")

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter="|", quoting=csv.QUOTE_ALL)
        w.writerow([
            "author",
            f"n_comments_{v}",
            f"pos_count_{v}",
            f"prev_{v}",
            f"mean_prob_{v}",
            f"std_prob_{v}",
        ])
        w.writerows(rows)

    print(f"[OK] Wrote {out_path}  (authors: {len(rows)})")

In [ ]:
# MAIN LOOP
def summarize_per_value_files():
    files = sorted(glob.glob(os.path.join(INPUT_DIR, FILE_GLOB)))
    if not files:
        raise FileNotFoundError(f"No files found in {INPUT_DIR} matching {FILE_GLOB}")

    for fp in files:
        print(f"\n=== Processing: {fp} ===")

        # Running aggregates per author for THIS value file
        # We keep count, positive count, sum(prob), sum(prob^2)
        agg = {}

        detected_value_name = None
        
        # Read in chunks with minimal columns to reduce memory
        usecols = ["id","author","value","pred","prob"]
        dtypes = {
            "id": "string",
            "author": "string",
            "value": "string",
            "pred": "int8",     
            "prob": "float32",  # probabilities
        }

        chunk_iter = pd.read_csv(
            fp,
            sep="|",
            usecols=usecols,
            dtype=dtypes,
            chunksize=CHUNK_SIZE,
            encoding="utf-8",
            engine="python",
            quoting=csv.QUOTE_MINIMAL, 
            on_bad_lines="skip"
        )

        for i, chunk in enumerate(chunk_iter, start=1):
            # Normalize author/value
            chunk["author"] = chunk["author"].fillna("")#.map(safe_upper_strip)
            chunk["value"] = chunk["value"].fillna("")#.map(safe_upper_strip)

            # Detect the value name once (assumes single value per file)
            if detected_value_name is None:
                # pick the most frequent non-empty value in this chunk
                vc = chunk["value"].value_counts()
                if not vc.empty:
                    detected_value_name = vc.index[0]

            # Drop rows with empty author
            chunk = chunk[chunk["author"] != ""]

            # Aggregate per author in this chunk
            gb = chunk.groupby("author", observed=True)
            size = gb.size()
            pos = gb["pred"].sum(min_count=1)
            sum_prob = gb["prob"].sum(min_count=1)
            sum_prob_sq = (chunk["prob"] ** 2).groupby(chunk["author"], observed=True).sum(min_count=1)

            # Merge into running aggregates
            for author in size.index:
                n_add = int(size.loc[author])
                pos_add = float(pos.loc[author]) if author in pos.index else 0.0
                sum_p_add = float(sum_prob.loc[author]) if author in sum_prob.index else 0.0
                sum_p2_add = float(sum_prob_sq.loc[author]) if author in sum_prob_sq.index else 0.0

                if author not in agg:
                    agg[author] = {"n": 0, "pos": 0.0, "sum_prob": 0.0, "sum_prob_sq": 0.0}

                agg[author]["n"] += n_add
                agg[author]["pos"] += pos_add
                agg[author]["sum_prob"] += sum_p_add
                agg[author]["sum_prob_sq"] += sum_p2_add

            if i % 20 == 0:
                print(f"  ...processed {i*CHUNK_SIZE:,} rows (running authors: {len(agg):,})")

        if detected_value_name is None:
            basename = os.path.basename(fp)
            detected_value_name = os.path.splitext(basename)[0].replace("deba_", "").replace("_usrs_comments", "").upper()
            print(f"  [WARN] Could not detect value from data; inferred from filename: {detected_value_name}")

        # Finalize stats and write summary for this value
        rows = finalize_stats(agg)
        write_summary(detected_value_name, rows, OUTPUT_DIR)

In [ ]:
if __name__ == "__main__":
    summarize_per_value_files()

In [ ]:
df_yamir = pd.read_csv("/Proyecto/Value-disagreement/Python/Models/Inference/deba_usr_comments/deba_achievement_usrs_comments.csv",sep="|")

In [ ]:
len(df_yamir)

In [ ]:
df_yamir[df_yamir['author']=='AMerrickanGirl']['id'].duplicated().any()

In [ ]:
df_yamir = pd.read_csv("/Proyecto/Value-disagreement/Python/Models/Inference/agg_by_valuefile/author_summary_achievement.csv",sep="|")

In [ ]:
len(df_yamir)

In [ ]:
df_yamir.n_comments_achievement.sum()

In [ ]:
df_yamir[df_yamir['author']=='AMerrickanGirl']

### ALL VALUES PROFILES

In [ ]:
# CONFIG
INPUT_DIR  = "/Proyecto/Value-disagreement/Python/Models/Inference/agg_by_valuefile"
OUTPUT_DIR = "/Proyecto/Value-disagreement/Python/Models/Inference/final_profiles"
FILE_GLOB = "author_summary_*.csv"

# Schwartz values (CIRCUMPLEX ORDER)
VALUES = Dict_Object.ValueConstants.SCHWARTZ_VALUES
VALUES = [VALUES[i] for i in Dict_Object.ValueConstants.SCHWARTZ_VALUES_CIRCUMPLEX_ORDER]

MIN_SUPPORT = 5#10

EXPORT_JSONL = True
JSONL_PATH   = "author_profiles.jsonl"

In [ ]:
# LOADING & MERGING
def infer_value_from_columns(cols):
    for prefix in ("n_comments_", "pos_count_", "prev_", "mean_prob_", "std_prob_"):
        for c in cols:
            if c.startswith(prefix):
                return c.replace(prefix, "")#.strip().upper()
    return None

In [ ]:
def load_value_summary(path):
    df = pd.read_csv(path, sep="|", dtype={"author":"string"}, encoding="utf-8", engine="python")
    df["author"] = df["author"]#.str.upper().str.strip()
    v = infer_value_from_columns(df.columns)
    if v is None:
        v = os.path.basename(path).replace("author_summary_", "").replace(".csv", "")#.upper()
    needed = [f"n_comments_{v}", f"pos_count_{v}", f"prev_{v}", f"mean_prob_{v}", f"std_prob_{v}"]
    for col in needed:
        if col not in df.columns: df[col] = np.nan
    return v, df[["author"] + needed].copy()

In [ ]:
def merge_all_values():
    paths = sorted(glob.glob(os.path.join(INPUT_DIR, FILE_GLOB)))
    if not paths:
        raise FileNotFoundError(f"No files found in {INPUT_DIR} matching {FILE_GLOB}")

    merged, present_values = None, []
    for p in paths:
        v, dfv = load_value_summary(p)
        present_values.append(v)
        merged = dfv if merged is None else merged.merge(dfv, on="author", how="outer")

    # Keep values in configured order
    present_values = sorted(set(present_values), key=lambda x: VALUES.index(x) if x in VALUES else 999)
    return merged, present_values

In [ ]:
def entropy_safe(mat):
    eps = 1e-12
    
    mat = np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)
    row_sum = mat.sum(axis=1, keepdims=True)
    
    norm = np.divide(mat, np.where(row_sum == 0, 1.0, row_sum))
    norm = np.clip(norm, eps, 1.0)
    
    return -np.sum(norm * np.log(norm), axis=1)

In [ ]:
"""def topk_labels(row, base, k=3):
    pairs = [(v, row[f"{base}_{v}"]) for v in values_in_data]
    pairs.sort(key=lambda x: x[1], reverse=True)
    return [v for v, _ in pairs[:k]]"""

def topk_labels(row, base, values_in_data, k=3, min_threshold=None, tie_break="pos_count"):
    """
    Select top-k labels for a row using the column prefix `base` (e.g., 'prev' or 'mean_prob').
    - Only values with score > min_threshold are considered (default: 0 for prev, 1e-6 for mean_prob).
    - Does NOT pad with zeros; returns fewer than k if fewer are expressed.
    - Deterministic tie-breaking via secondary key:
        tie_break='pos_count' (default), 'n_comments', or 'alpha' (value name).
    """
    # sensible defaults per base
    if min_threshold is None:
        min_threshold = 0.0 if base == "prev" else 1e-6

    # collect (value, score, secondary) triples
    triples = []
    for v in values_in_data:
        score = row.get(f"{base}_{v}", 0.0)
        if score is None:
            score = 0.0
        if score > min_threshold:
            if tie_break == "pos_count":
                secondary = row.get(f"pos_count_{v}", 0)
            elif tie_break == "n_comments":
                secondary = row.get(f"n_comments_{v}", 0)
            elif tie_break == "alpha":
                secondary = -ord(v[0])  # simple deterministic fallback
            else:
                secondary = 0
            triples.append((v, float(score), float(secondary)))

    if not triples:
        return []  # nothing expressed

    # sort by (score desc, secondary desc), then take top-k
    triples.sort(key=lambda t: (t[1], t[2]), reverse=True)
    return [v for v, _, _ in triples[:k]]

In [ ]:
# WIDE TABLE
def build_wide_table(merged_df, values_in_data):
    df = merged_df.copy()

    ordered = [v for v in VALUES if v in values_in_data] + [v for v in values_in_data if v not in VALUES]
    
    n_all = df[[f"n_comments_{v}" for v in values_in_data]].sum(axis=1)
    df.insert(1, "n_comments_all", n_all.astype("int64"))

    # value mention per value
    pos_cols = [f"pos_count_{v}" for v in values_in_data]
    df.insert(2, "total_value_mentions", df[pos_cols].sum(axis=1).astype("int64"))
    
    for v in values_in_data:
        df[f"n_comments_{v}"] = df[f"n_comments_{v}"].fillna(0).astype("int64")
        df[f"pos_count_{v}"]  = df[f"pos_count_{v}"].fillna(0).astype("int64")
        df[f"prev_{v}"]       = df[f"prev_{v}"].fillna(0.0).astype("float32")
        df[f"mean_prob_{v}"]  = df[f"mean_prob_{v}"].fillna(0.0).astype("float32")
        df[f"std_prob_{v}"]   = df[f"std_prob_{v}"].fillna(0.0).astype("float32")
        df[f"paper_profile_{v}"] = np.where(df["total_value_mentions"] > 0, 
                                            df[f"pos_count_{v}"] / df["total_value_mentions"],
                                            0.0).astype("float32")
    
    # Matrices
    prev_matrix = df[[f"prev_{v}" for v in values_in_data]].to_numpy()
    mean_mat = df[[f"mean_prob_{v}" for v in ordered]].to_numpy(dtype="float64")
    paper_mat = df[[f"paper_profile_{v}" for v in ordered]].to_numpy()
    
    diversity = (prev_matrix > 0).sum(axis=1)
    df.insert(3, "diversity_values_expressed", diversity.astype("int16"))

    # Top 3 values binary
    top3_binary = df.apply(lambda r: ",".join(topk_labels(r, base="prev", values_in_data=values_in_data, k=3, min_threshold=0.0, 
                                                          tie_break="pos_count")), axis=1)
    df.insert(4, "top3_binary_values", top3_binary)

    # Top 3 values proba
    top3_prob   = df.apply(lambda r: ",".join(topk_labels(r, base="mean_prob", values_in_data=values_in_data, k=3, min_threshold=0.0, 
                                                          tie_break="pos_count")), axis=1)
    df.insert(5, "top3_prob_values",   top3_prob)

    # Top 3 values paper profile
    top3_paper = df.apply(lambda r: ",".join(topk_labels(r,base="paper_profile",values_in_data=values_in_data,k=3,
                                                         min_threshold=0.0,tie_break="pos_count")),axis=1)
    df.insert(6, "top3_paper_values", top3_paper)

    # Entropy binary
    entropy_prev = entropy_safe(prev_matrix)
    df.insert(7, "entropy_binary_prev", entropy_prev.astype("float32"))

    # Entropy Proba
    entropy_mean = entropy_safe(mean_mat)
    df.insert(8, "entropy_mean_prob_profile", entropy_mean)

    # Entropy Paper profile
    entropy_paper = entropy_safe(paper_mat)
    df.insert(9, "entropy_paper_profile", entropy_paper)

    df.insert(10, "well_supported", (df["n_comments_all"] >= MIN_SUPPORT).astype("int8"))

    meta_cols = ["author","n_comments_all","total_value_mentions","diversity_values_expressed",
                "top3_binary_values","top3_prob_values","top3_paper_values",
                "entropy_binary_prev","entropy_mean_prob_profile","entropy_paper_profile",
                "well_supported"]

    counts_cols = [f"n_comments_{v}" for v in ordered]
    pos_cols    = [f"pos_count_{v}" for v in ordered]
    prev_cols   = [f"prev_{v}" for v in ordered]
    mean_cols   = [f"mean_prob_{v}" for v in ordered]
    std_cols    = [f"std_prob_{v}" for v in ordered]
    paper_cols = [f"paper_profile_{v}" for v in ordered]
    
    df = df[meta_cols + counts_cols + pos_cols + prev_cols + mean_cols + std_cols + paper_cols]

    df = df.sort_values("n_comments_all", ascending=False, kind="stable").reset_index(drop=True)
    return df, ordered

In [ ]:
# COMPACT (VECTOR) TABLE
def build_compact_table(df_wide, ordered_values):
    df = df_wide.copy()

    def row_vector(row, prefix):
        return [row[f"{prefix}_{v}"] for v in ordered_values]

    vectors = {
        "n_comments_vector": [],
        "pos_count_vector": [],
        "prev_vector": [],
        "mean_prob_vector": [],
        "std_prob_vector": [],
        "paper_profile_vector": []        
    }

    for _, r in df.iterrows():
        vectors["n_comments_vector"].append(row_vector(r, "n_comments"))
        vectors["pos_count_vector"].append(row_vector(r, "pos_count"))
        vectors["prev_vector"].append(row_vector(r, "prev"))
        vectors["mean_prob_vector"].append(row_vector(r, "mean_prob"))
        vectors["std_prob_vector"].append(row_vector(r, "std_prob"))
        vectors["paper_profile_vector"].append(row_vector(r, "paper_profile"))

    compact = pd.DataFrame({
        "author": df["author"],
        "n_comments_all": df["n_comments_all"],
        "total_value_mentions": df["total_value_mentions"].astype("int64"),
        "diversity_values_expressed": df["diversity_values_expressed"],
        "top3_binary_values": df["top3_binary_values"].str.split(","),
        "top3_prob_values": df["top3_prob_values"].str.split(","),
        "top3_paper_values": df["top3_paper_values"].str.split(","),
        "entropy_binary_prev": df["entropy_binary_prev"],
        "entropy_paper_profile": df["entropy_paper_profile"].astype("float32"),
        "entropy_mean_prob_profile": df["entropy_mean_prob_profile"].astype("float32"),
        "well_supported": df["well_supported"],
        "values_order": [ordered_values]*len(df),  # to make vector alignment explicit
        "n_comments_vector": vectors["n_comments_vector"],
        "pos_count_vector": vectors["pos_count_vector"],
        "prev_vector": vectors["prev_vector"],
        "mean_prob_vector": vectors["mean_prob_vector"],
        "std_prob_vector": vectors["std_prob_vector"],
        "paper_profile_vector": vectors["paper_profile_vector"],
    })

    compact_csv = compact.copy()
    list_cols = ["top3_binary_values","top3_prob_values","top3_paper_values",
                 "values_order","n_comments_vector","pos_count_vector","prev_vector",
                 "mean_prob_vector","std_prob_vector","paper_profile_vector"]
    
    for c in list_cols:
        compact_csv[c] = compact_csv[c].apply(lambda x: json.dumps(x, ensure_ascii=False))

    return compact, compact_csv

In [ ]:
# SAVE
def save_all(df_wide, df_compact, df_compact_csv, values_in_data):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Wide
    out_wide_csv = os.path.join(OUTPUT_DIR, "author_profiles_wide.csv")
    df_wide.to_csv(out_wide_csv, sep="|", index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")
    print(f"[OK] Wrote {out_wide_csv}  (authors: {len(df_wide):,})")

    try:
        out_wide_parquet = os.path.join(OUTPUT_DIR, "author_profiles_wide.parquet")
        df_wide.to_parquet(out_wide_parquet, index=False)
        print(f"[OK] Wrote {out_wide_parquet}")
    except Exception as e:
        print(f"[WARN] Parquet (wide) skipped: {e}")

    # Compact (CSV with JSON-ified lists)
    out_compact_csv = os.path.join(OUTPUT_DIR, "author_profiles_compact.csv")
    df_compact_csv.to_csv(out_compact_csv, sep="|", index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")
    print(f"[OK] Wrote {out_compact_csv}")

    # Compact Parquet (keeps lists as arrays if pyarrow available)
    try:
        out_compact_parquet = os.path.join(OUTPUT_DIR, "author_profiles_compact.parquet")
        df_compact.to_parquet(out_compact_parquet, index=False)
        print(f"[OK] Wrote {out_compact_parquet}")
    except Exception as e:
        print(f"[WARN] Parquet (compact) skipped: {e}")

    # JSONL (compact)
    if EXPORT_JSONL:
        out_jsonl = os.path.join(OUTPUT_DIR, "author_profiles_compact.jsonl")
        with open(out_jsonl, "w", encoding="utf-8") as f:
            for _, row in df_compact.iterrows():
                rec = row.to_dict()
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print(f"[OK] Wrote {out_jsonl}")

    with open(os.path.join(OUTPUT_DIR, "values_used.json"), "w", encoding="utf-8") as f:
        json.dump(values_in_data, f, ensure_ascii=False, indent=2)
        print("[OK] Wrote values_used.json")

In [ ]:
# MAIN
def main():
    merged, present_values = merge_all_values()
    print(f"Detected {len(present_values)} value summaries: {present_values}")

    df_wide, ordered_values = build_wide_table(merged, present_values)
    df_compact, df_compact_csv = build_compact_table(df_wide, ordered_values)
    save_all(df_wide, df_compact, df_compact_csv, ordered_values)

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
df_wide = pd.read_csv("/Proyecto/Value-disagreement/Python/Models/Inference/final_profiles/author_profiles_wide.csv",sep="|")

In [ ]:
df_wide

In [ ]:
df_compact = pd.read_csv("/Proyecto/Value-disagreement/Python/Models/Inference/final_profiles/author_profiles_compact.csv",sep="|")

In [ ]:
df_compact

In [ ]:
# RESULT TABLES
df_compact['author'].duplicated().any() 

In [ ]:
# table 55
df = df_compact  

table_55 = pd.DataFrame({
    "Métrica": [
        "Usuarios totales",
        "Usuarios con valores detectados",
        "Media comentarios por usuario",
        "Media total_value_mentions",
        "Mediana total_value_mentions",
        "% usuarios con perfil válido"
    ],
    "Valor": [
        len(df),
        (df["total_value_mentions"] > 0).sum(),
        df["n_comments_all"].mean(),
        df["total_value_mentions"].mean(),
        df["total_value_mentions"].median(),
        (df["total_value_mentions"] > 0).mean() * 100
    ],
})

In [ ]:
table_55

In [ ]:
# table 56
df_wide

In [ ]:
value_cols = [c for c in df_wide.columns if c.startswith("mean_prob_")]
value_cols

In [ ]:
table_56 = (
    df_wide[value_cols]
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={"index": "Valor"})
)
table_56

In [ ]:
table_A2 = df_wide[[
    "entropy_binary_prev",
    "entropy_mean_prob_profile",
    "entropy_paper_profile"
]].agg(["mean", "std"]).T.reset_index()

In [ ]:
table_A2